# 07 · Quantum Algorithms In-Depth

You've met the qubit (01), the gate set (02), variational QML (03), the classical
foundations (04), the real hardware (05), and the road ahead (06). This notebook goes
**deeper into the engine room**: the *algorithms* that give quantum computers their
theoretical power, the **mathematics** underneath them, and an honest, sourced snapshot
of where real machines stand.

We cover:

1. **Complexity classes** (P, BPP, BQP) — what "quantum advantage" *formally* means
2. The **math**: amplitudes, interference, and the Bloch sphere
3. The **Quantum Fourier Transform (QFT)** — the workhorse subroutine
4. **Quantum Phase Estimation (QPE)** — reading eigenvalues
5. **Shor's algorithm** — factoring, and the post-quantum-cryptography scramble
6. **Grover's search** — a quadratic speedup, with a *runnable* Qiskit demo
7. **Variational algorithms** — VQE & QAOA, the NISQ-era workhorses
8. **Quantum error correction in depth** — surface codes vs. qLDPC
9. **State of the field (June 2026)** — verified numbers

> This notebook is more advanced than 01-03 but still self-contained. Everything runs on
> a local **Qiskit 1.x** simulator. Time-sensitive figures are current as of **June 2026** —
> check the cited sources for the latest.


## Setup

In [ ]:
# Run once. Comment out after the first run.
%pip install qiskit qiskit-aer matplotlib pylatexenc

## 1. Complexity classes: what "quantum advantage" really means

Computer scientists sort problems into **complexity classes** by how the effort to solve
them grows with input size. Three matter for quantum computing:

| Class | Meaning | Example |
|-------|---------|---------|
| **P** | Solvable *efficiently* (polynomial time) on a classical computer | Sorting, shortest path |
| **BPP** | Efficiently solvable by a classical computer allowed *randomness*, with bounded error | Primality testing |
| **BQP** | **B**ounded-error **Q**uantum **P**olynomial time — efficiently solvable by a *quantum* computer | Factoring (Shor), quantum simulation |

The big open question is whether **BQP is strictly larger than BPP** — i.e. whether there
are problems quantum computers solve efficiently that classical ones provably cannot. It is
*widely believed* but **not proven** (it would imply `P ≠ PSPACE`, a famous open problem).

Two terms people often confuse:

- **Quantum advantage / supremacy** — an *experimental* claim: a real device beats the best
  classical computers *today* on some task (e.g. Google's Random Circuit Sampling).
- **BQP membership** — an *asymptotic* claim about how the problem scales as size → ∞.

A task can show experimental "advantage" without being practically useful — which is exactly
why Google's later **verifiable** advantage (notebook 05) mattered so much.


## 2. The math: amplitudes, interference, and the Bloch sphere

A single qubit is a unit vector in a 2-dimensional complex space:

$$|\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle,\qquad |\alpha|^2 + |\beta|^2 = 1$$

The amplitudes $\alpha,\beta$ are **complex numbers** — and that complex *phase* is the secret
sauce. Classical probabilities are non-negative and only ever add. Amplitudes can be negative
or complex, so they can **cancel** (destructive interference) or **reinforce** (constructive
interference). *Every* quantum algorithm is, at heart, a recipe to arrange interference so that
**wrong answers cancel and right answers add up**.

Because global phase is unobservable, a single qubit's state maps onto the surface of a sphere —
the **Bloch sphere** — with two angles:

$$|\psi\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|1\rangle$$

- $\theta$ (polar angle) sets the measurement probabilities,
- $\varphi$ (azimuth) is the **relative phase** — invisible to a single measurement but crucial
  once gates and interference come into play.

`n` qubits live in a $2^n$-dimensional space, which is why simulating them classically blows up.


## 3. The Quantum Fourier Transform (QFT)

The **QFT** is the quantum analogue of the discrete Fourier transform — it maps amplitudes
into the "frequency" domain. It is the key subroutine behind phase estimation and Shor's
algorithm.

The magic is efficiency: the classical Fast Fourier Transform on $2^n$ values costs
$O(n\,2^n)$ operations, but the **QFT needs only $O(n^2)$ gates**. (You can't read all the
amplitudes out directly — but when an algorithm only needs the *interference pattern*, that
exponential speedup is real.)

Let's build a 3-qubit QFT circuit and look at its structure: Hadamards interleaved with
**controlled phase rotations**, the hallmark of the transform.


In [ ]:
from qiskit.circuit.library import QFT

qft = QFT(num_qubits=3)
qft.decompose().draw("mpl")

## 4. Quantum Phase Estimation (QPE)

Most quantum speedups trace back to one primitive: **Quantum Phase Estimation**. Given a
unitary $U$ and one of its eigenstates $|\psi\rangle$ with $U|\psi\rangle = e^{2\pi i\theta}|\psi\rangle$,
QPE estimates the phase $\theta$.

The trick is **phase kickback** + the **inverse QFT**: a register of "counting" qubits picks up
the phase through controlled applications of $U$, and the inverse QFT turns that phase into a
binary number you can measure.

QPE powers:

- **Shor's algorithm** (the period-finding core),
- **HHL** for linear systems,
- **quantum chemistry** (reading off molecular energy levels).

The catch: textbook QPE needs **deep, error-corrected circuits**, so it is largely *beyond*
today's NISQ machines. That limitation is the entire reason the **variational** algorithms in
§7 exist — they trade QPE's exactness for shallow, noise-tolerant circuits.


## 5. Shor's algorithm — and why the world is rushing to post-quantum crypto

In 1994 Peter Shor showed a quantum computer can **factor large integers** and compute
**discrete logarithms** in *polynomial* time — roughly $O(n^3)$ for an $n$-bit number — using
period-finding built on QPE and the QFT.

Why it shook the world: the security of **RSA** and **elliptic-curve** cryptography rests on
factoring/discrete-log being *classically* infeasible. The best classical method (the General
Number Field Sieve) is **sub-exponential**; Shor is **polynomial**. The plot below shows why
that gap is existential for current cryptography.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bits = np.arange(256, 4097, 256)   # integer (key) sizes in bits

# Classical General Number Field Sieve ~ exp(1.9 * L^(1/3) * (ln L)^(2/3)), L = n*ln2
def gnfs_ops(n_bits):
    L = n_bits * np.log(2)
    return np.exp(1.9 * (L ** (1/3)) * (np.log(L) ** (2/3)))

# Shor's algorithm is polynomial, ~ n^3
def shor_ops(n_bits):
    return n_bits ** 3.0

plt.figure(figsize=(8, 4.5))
plt.semilogy(bits, gnfs_ops(bits), "o-", label="Classical GNFS (sub-exponential)")
plt.semilogy(bits, shor_ops(bits), "s-", label="Shor's algorithm (polynomial)")
plt.xlabel("Key size (bits)"); plt.ylabel("Approx. operations (log scale)")
plt.title("Why Shor matters: factoring effort vs. key size")
plt.legend(); plt.tight_layout(); plt.show()

print(f"RSA-2048: classical ~{gnfs_ops(2048):.1e} ops   vs   Shor ~{shor_ops(2048):.1e} ops")

**Reality check (June 2026):** Shor's algorithm is *strategically* decisive but **not yet
practical** — factoring RSA-2048 is estimated to need on the order of *millions* of high-quality
physical qubits with full error correction, which no one has. But the threat is taken seriously
*now*: the U.S. NIST has standardized **post-quantum cryptography** (e.g. CRYSTALS-Kyber and
Dilithium), and "harvest-now, decrypt-later" is driving migration today. This is the clearest
real-world consequence of quantum computing so far — even before the hardware exists.


## 6. Grover's algorithm — quadratic search (runnable demo)

Lov Grover's 1996 algorithm finds a marked item in an **unstructured** search space of size
$N$ in about $\frac{\pi}{4}\sqrt{N}$ steps, versus $N/2$ on average classically — a **quadratic**
speedup. It works by *amplitude amplification*: an **oracle** flips the phase of the target
state, then a **diffuser** reflects all amplitudes about their mean, nudging probability toward
the target. Repeat $\sim\!\sqrt{N}$ times.

First, see how the iteration count grows far slower than brute force:


In [ ]:
import numpy as np

print(f"{'qubits':>6} | {'search space N':>16} | {'classical (avg)':>16} | {'Grover iters':>12}")
print("-" * 60)
for n in [2, 4, 8, 16, 20, 30]:
    N = 2 ** n
    grover_iters = int(np.floor(np.pi / 4 * np.sqrt(N)))
    print(f"{n:>6} | {N:>16,} | {N // 2:>16,} | {grover_iters:>12,}")

Now a real 2-qubit Grover circuit that searches for the marked state $|11\rangle$. For 2 qubits
($N=4$), **one** Grover iteration is optimal and finds the answer with certainty.


In [ ]:
from qiskit import QuantumCircuit

grover = QuantumCircuit(2)
grover.h([0, 1])                    # 1. uniform superposition over 00,01,10,11

# 2. Oracle: flip the PHASE of the marked state |11>  (CZ does exactly this)
grover.cz(0, 1)

# 3. Diffuser: invert all amplitudes about their mean (amplifies the marked state)
grover.h([0, 1]); grover.x([0, 1]); grover.cz(0, 1); grover.x([0, 1]); grover.h([0, 1])

grover.measure_all()
grover.draw("mpl")

In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

counts = StatevectorSampler().run([grover], shots=1024).result()[0].data.meas.get_counts()
print("Grover result:", counts)   # essentially all '11'
plot_histogram(counts)

The marked state $|11\rangle$ comes up ~100% of the time — the oracle + diffuser amplified it
in a single shot. Note the speedup is "only" quadratic, so Grover does **not** break symmetric
crypto outright; it effectively halves key strength (which is why AES-256 stays safe while
AES-128 is weakened).


## 7. Variational algorithms: VQE & QAOA (the NISQ workhorses)

Since QPE-based algorithms need fault-tolerant hardware we don't have yet, today's machines lean
on **variational quantum algorithms** — the same hybrid quantum-classical loop you trained in
notebook 03:

- **VQE (Variational Quantum Eigensolver):** finds the *lowest eigenvalue* (ground-state energy)
  of a Hamiltonian. Prepare a trial state with a parameterized **ansatz**, measure its energy,
  and let a **classical optimizer** lower it. This is the leading near-term approach to
  **quantum chemistry** and materials.
- **QAOA (Quantum Approximate Optimization Algorithm):** the same idea aimed at **combinatorial
  optimization** (routing, scheduling, portfolios) — alternating "cost" and "mixer" layers whose
  angles a classical optimizer tunes.

Both trade QPE's exactness for **shallow, noise-tolerant circuits**. Here is VQE *in miniature*:
a one-qubit Hamiltonian $H = Z$ whose ground state is $|1\rangle$ with energy $-1$. We sweep the
single ansatz angle and let the lowest point stand in for the classical optimizer.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

H = SparsePauliOp("Z")   # toy Hamiltonian; ground state |1>, energy -1

def energy(theta):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)                                   # ansatz: one trainable rotation
    return Statevector(qc).expectation_value(H).real  # <psi|H|psi>

thetas = np.linspace(0, 2 * np.pi, 200)
landscape = [energy(t) for t in thetas]

best_theta = thetas[int(np.argmin(landscape))]
print(f"VQE estimate: ground-state energy ~ {min(landscape):.3f} "
      f"at theta = {best_theta:.3f}  (exact: -1 at theta = pi = {np.pi:.3f})")

plt.plot(thetas, landscape)
plt.axhline(-1, ls="--", color="gray", label="true ground energy")
plt.scatter([best_theta], [min(landscape)], color="red", zorder=5, label="VQE minimum")
plt.xlabel("ansatz angle theta"); plt.ylabel(r"energy  $\langle H\rangle=\langle Z\rangle$")
plt.title("VQE in miniature: minimize energy over the ansatz parameter")
plt.legend(); plt.tight_layout(); plt.show()

Scale this up — many qubits, a Hamiltonian built from a real molecule, and a proper optimizer —
and you have the template behind essentially every near-term quantum-chemistry result. The
honest caveat from notebook 06 still applies: **barren plateaus** can flatten the landscape as
circuits grow, so ansatz design is an active research area.


## 8. Quantum error correction in depth

A logical, reliable qubit is built from many noisy physical ones. Two code families dominate the
2026 conversation:

- **Surface codes** — the long-time front-runner (Google Willow, IBM Heron). Robust and only
  needs nearest-neighbour connectivity, but **expensive**: roughly **121 physical qubits per
  logical qubit at distance 11**, and ~1,000 once you include routing/overhead. Threshold ≈ 1%.
- **qLDPC codes (quantum low-density parity-check)** — the rising challenger. By using
  longer-range connectivity they pack far more logical qubits into the same hardware. IBM's
  bivariate-bicycle ("gross") code targets ~**10× less** overhead; in June 2026 **IQM** reported
  "barbell" qLDPC codes needing up to **8× fewer** physical qubits than surface codes while
  cutting logical error rates by up to **1,000×**.

The trade-off: qLDPC needs each qubit to talk to many others (harder to wire up), while surface
codes are simpler but hungrier. The overhead numbers below are why this race matters — they
decide *when* fault-tolerant machines become buildable.


In [ ]:
rows = [
    ("Surface code (distance 11)",      "~121 physical / logical"),
    ("Surface code (with routing)",     "~1,000 physical / logical"),
    ("IBM qLDPC 'gross' code",          "~10x fewer than surface"),
    ("IQM 'barbell' qLDPC (Jun 2026)",  "up to 8x fewer than surface"),
]
print(f"{'Code':32s} {'Overhead':>28s}")
print("-" * 62)
for name, overhead in rows:
    print(f"{name:32s} {overhead:>28s}")

## 9. State of the field (June 2026)

A sourced snapshot — verify the linked sources for the latest, since these move fast:

| Milestone | Figure (June 2026) |
|-----------|--------------------|
| Largest verified **logical**-qubit count | **QuEra ~96** logical qubits (neutral atoms); **Quantinuum Helios ~48** (trapped ion) |
| Best **2-qubit gate fidelity** | IonQ & Silicon Quantum Computing **99.99%**; Quantinuum 99.97% |
| Google **Willow** | **105** physical qubits, below-threshold QEC; verifiable advantage (~13,000×) |
| IBM near-term | **Nighthawk** (120 qubits); **Kookaburra** module (qLDPC memory) targeted 2026 |
| Quantum-volume record | Quantinuum H2: **2^25** (Sept 2025) |
| Fault-tolerance targets | IBM **Starling** ~2029; Quantinuum / Microsoft also ~2029-2030 |

> **Note on notebook 05:** that notebook quoted Willow as "65-qubit" — the correct, published
> figure is **105 qubits** (Google, Dec 2024). The number above is the corrected one.

**Bottom line:** we are in the **late-NISQ era**. Shor and QPE are proven on paper but await
fault-tolerant hardware; Grover gives a modest quadratic edge; and **variational algorithms
(VQE/QAOA)** plus rapidly improving **error correction** are where the practical action is right now.


## Summary

- **Complexity:** quantum power is the (believed, unproven) gap **BQP ⊋ BPP**; "advantage" is an
  experimental claim, distinct from asymptotic scaling.
- **Mechanism:** algorithms engineer **interference** so wrong answers cancel — phase is everything.
- **QFT → QPE** is the backbone of the exponential speedups (**Shor**, chemistry, HHL).
- **Shor** (polynomial factoring) threatens RSA/ECC → driving **post-quantum cryptography** today.
- **Grover** gives a **quadratic** search speedup (runnable above).
- **VQE & QAOA** are the **NISQ-era** workhorses — shallow, hybrid, noise-tolerant.
- **Error correction** (surface vs. **qLDPC**) overhead decides when fault tolerance arrives.

🎉 You've now gone from subatomic particles all the way to the algorithms and the 2026 frontier.

### References
- Wikipedia — *Shor's algorithm*: https://en.wikipedia.org/wiki/Shor's_algorithm
- Fortinet — *Shor's & Grover's algorithms / encryption threat*: https://www.fortinet.com/resources/cyberglossary/shors-grovers-algorithms
- IBM Quantum Learning — *Variational quantum algorithms*: https://quantum.cloud.ibm.com/learning/en/courses/utility-scale-quantum-computing/variational-quantum-algorithms
- Quandela — *Variational Quantum Eigensolver (VQE)*: https://www.quandela.com/resources/quantum-computing-glossary/variational-quantum-eigensolver-vqe-explained/
- Wikipedia — *Quantum supremacy* (BQP, advantage): https://en.wikipedia.org/wiki/Quantum_supremacy
- Google — *Meet Willow* (105 qubits): https://blog.google/innovation-and-ai/technology/research/google-willow-quantum-chip/
- The Quantum Insider — *IQM qLDPC codes (2026)*: https://thequantuminsider.com/2026/06/23/iqm-says-new-quantum-codes-cut-logical-error-rates-by-up-to-1000-times/
- PennyLane — *qLDPC codes for quantum error correction*: https://pennylane.ai/demos/tutorial_qldpc_codes
- Quantinuum — *accelerated fault-tolerance roadmap*: https://www.quantinuum.com/press-releases/quantinuum-unveils-accelerated-roadmap-to-achieve-universal-fault-tolerant-quantum-computing-by-2030
